### CONEXION DDBB OLIST

In [1]:
%pip install PyMySQL
from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)



Note: you may need to restart the kernel to use updated packages.
Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
import pandas as pd
from IPython.display import display, Markdown

# --- Cargar datos base (añadimos ciudad, estado y categoría) ---
query = """
SELECT 
    c.customer_city,
    c.customer_state,
    pct.product_category_name_english AS product_category_name,
    i.price,
    o.order_id,
    o.order_purchase_timestamp
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p
    ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
LEFT JOIN olist_orders_dataset o
    ON i.order_id = o.order_id
LEFT JOIN olist_customers_dataset c
    ON o.customer_id = c.customer_id
WHERE o.order_status <> 'canceled'
"""
df = pd.read_sql_query(query, con=engine)

# --- Procesamiento temporal ---
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df["order_year"] = df["order_purchase_timestamp"].dt.year
df = df[df["order_year"].isin([2017, 2018])]

# --- Agrupar por ciudad, estado, categoría y año ---
ventas_por_ciudad_cat = (
    df.groupby(["customer_city", "customer_state", "product_category_name", "order_year"], dropna=False)
      .agg(
          TotalSales=("price", "sum"),
          OrdersQty=("order_id", "nunique")
      )
      .reset_index()
)

# --- Añadir nombres completos de los estados brasileños ---
state_map = {
    'AC': 'Acre, Brazil', 'AL': 'Alagoas, Brazil', 'AM': 'Amazonas, Brazil', 'AP': 'Amapá, Brazil',
    'BA': 'Bahia, Brazil', 'CE': 'Ceará, Brazil', 'DF': 'Distrito Federal, Brazil', 'ES': 'Espírito Santo, Brazil',
    'GO': 'Goiás, Brazil', 'MA': 'Maranhão, Brazil', 'MG': 'Minas Gerais, Brazil', 'MS': 'Mato Grosso do Sul, Brazil',
    'MT': 'Mato Grosso, Brazil', 'PA': 'Pará, Brazil', 'PB': 'Paraíba, Brazil', 'PE': 'Pernambuco, Brazil',
    'PI': 'Piauí, Brazil', 'PR': 'Paraná, Brazil', 'RJ': 'Rio de Janeiro, Brazil', 'RN': 'Rio Grande do Norte, Brazil',
    'RO': 'Rondônia, Brazil', 'RR': 'Roraima, Brazil', 'RS': 'Rio Grande do Sul, Brazil', 'SC': 'Santa Catarina, Brazil',
    'SE': 'Sergipe, Brazil', 'SP': 'São Paulo, Brazil', 'TO': 'Tocantins, Brazil'
}

ventas_por_ciudad_cat["state_fullname"] = ventas_por_ciudad_cat["customer_state"].map(state_map)

# ---  Añadir campo de país ---
ventas_por_ciudad_cat["country"] = "Brazil"

# ---  Redondear y ordenar ---
ventas_por_ciudad_cat["TotalSales"] = ventas_por_ciudad_cat["TotalSales"].round(2)
ventas_por_ciudad_cat = ventas_por_ciudad_cat.sort_values(by="TotalSales", ascending=False)

# ---  Mostrar resultados detallados ---
display(Markdown("###  Ventas totales por ciudad, estado, categoría y año"))
display(ventas_por_ciudad_cat.head(20))

# ---  Totales globales por año ---
total_sales_by_year = (
    ventas_por_ciudad_cat.groupby("order_year")["TotalSales"]
    .sum()
    .reset_index(name="TotalSalesGlobal")
)
display(Markdown("### Totales globales por año"))
display(total_sales_by_year)

# ---  Agrupar por estado, categoría y año (para mapa) ---
ventas_por_estado = (
    ventas_por_ciudad_cat.groupby(
        ["country", "customer_state", "state_fullname", "product_category_name", "order_year"],
        dropna=False
    )
    .agg(
        TotalSales=("TotalSales", "sum"),
        OrdersQty=("OrdersQty", "sum")
    )
    .reset_index()
    .sort_values(by=["order_year", "TotalSales"], ascending=[True, False])
)

# ---  Mostrar resumen por estado y categoría ---
display(Markdown("###  Ventas totales por estado, categoría y año (para mapa)"))
display(ventas_por_estado.head(20))


###  Ventas totales por ciudad, estado, categoría y año

,customer_city,customer_state,product_category_name,order_year,TotalSales,OrdersQty,state_fullname,country
28186,sao paulo,SP,health_beauty,2018,131691.90,1029,"São Paulo, Brazil",Brazil
28236,sao paulo,SP,watches_gifts,2018,102293.73,496,"São Paulo, Brazil",Brazil
28116,sao paulo,SP,bed_bath_table,2018,98206.72,949,"São Paulo, Brazil",Brazil
28131,sao paulo,SP,computers_accessories,2018,85822.66,641,"São Paulo, Brazil",Brazil
28226,sao paulo,SP,sports_leisure,2018,81568.42,724,"São Paulo, Brazil",Brazil
28198,sao paulo,SP,housewares,2018,72343.11,688,"São Paulo, Brazil",Brazil
28115,sao paulo,SP,bed_bath_table,2017,72194.25,697,"São Paulo, Brazil",Brazil
28235,sao paulo,SP,watches_gifts,2017,61580.84,277,"São Paulo, Brazil",Brazil
28225,sao paulo,SP,sports_leisure,2017,61164.35,525,"São Paulo, Brazil",Brazil
28130,sao paulo,SP,computers_accessories,2017,56628.33,355,"São Paulo, Brazil",Brazil


### Totales globales por año

,order_year,TotalSalesGlobal
0,2017,6108492.27
1,2018,7341182.41


###  Ventas totales por estado, categoría y año (para mapa)

,country,customer_state,state_fullname,product_category_name,order_year,TotalSales,OrdersQty
2205,Brazil,SP,"São Paulo, Brazil",bed_bath_table,2017,207392.78,1925
2328,Brazil,SP,"São Paulo, Brazil",watches_gifts,2017,176468.44,816
2318,Brazil,SP,"São Paulo, Brazil",sports_leisure,2017,168014.89,1447
2276,Brazil,SP,"São Paulo, Brazil",health_beauty,2017,157016.53,1286
2220,Brazil,SP,"São Paulo, Brazil",computers_accessories,2017,136248.51,968
2268,Brazil,SP,"São Paulo, Brazil",furniture_decor,2017,135387.91,1376
2230,Brazil,SP,"São Paulo, Brazil",cool_stuff,2017,128454.40,756
2326,Brazil,SP,"São Paulo, Brazil",toys,2017,106367.38,934
2288,Brazil,SP,"São Paulo, Brazil",housewares,2017,96210.48,1126
2308,Brazil,SP,"São Paulo, Brazil",perfumery,2017,85115.79,646


In [3]:
# Total global esperado
total_global = df["price"].sum().round(2)
print(f" Total global (sum(price)): {total_global:,.2f}")

# Totales por estado y año (lo mismo que usa Looker)
check_estado = (
    ventas_por_estado.groupby(["state_fullname", "order_year"])["TotalSales"]
    .sum()
    .reset_index()
    .sort_values(["order_year", "TotalSales"], ascending=[True, False])
)
display(check_estado.head(20))

# Total agrupado (debería coincidir con el global)
total_estado = check_estado["TotalSales"].sum().round(2)
print(f" Total agrupado por estado: {total_estado:,.2f}")
print(f" Diferencia: {abs(total_global - total_estado):,.2f}")


 Total global (sum(price)): 13,449,674.68


,state_fullname,order_year,TotalSales
50,"São Paulo, Brazil",2017,2190452.45
40,"Rio de Janeiro, Brazil",2017,901158.52
24,"Minas Gerais, Brazil",2017,717852.50
38,"Rio Grande do Sul, Brazil",2017,356704.72
26,"Paraná, Brazil",2017,294101.24
8,"Bahia, Brazil",2017,234056.77
46,"Santa Catarina, Brazil",2017,232303.51
16,"Goiás, Brazil",2017,136535.45
12,"Distrito Federal, Brazil",2017,133580.64
32,"Pernambuco, Brazil",2017,122594.96


 Total agrupado por estado: 13,449,674.68
 Diferencia: 0.00


In [ ]:
from sqlalchemy import text
from sqlalchemy.types import Float, Integer, String

# ---  Guardar DataFrame en la base de datos ---
table_name = "dash_olist_states"

# ---  Eliminar tabla si existe (para evitar conflictos por reflexión) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))

# ---  Definir tipos de columnas ---
dtype_map = {
    "country": String(50),
    "customer_state": String(10),
    "state_fullname": String(100),
    "product_category_name": String(100),
    "order_year": Integer(),
    "TotalSales": Float(),
    "OrdersQty": Integer(),
}

# ---  Crear y cargar tabla ---
ventas_por_estado.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",    
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f" Tabla '{table_name}' creada y cargada correctamente en la base de datos.")

# ---  Validación opcional: mostrar número de registros cargados ---
with engine.connect() as conn:
    result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`;"))
    total_rows = result.scalar()
    print(f" Total de registros insertados: {total_rows:,}")


 Tabla 'dash_olist_states' creada y cargada correctamente en la base de datos.
 Total de registros insertados: 2,392
